"""ETL job to create modeling datasets from intermediates for collaborative-filtering."""

In [0]:
%run ../../config/utils

In [0]:
from datetime import datetime
import os
import sys
sys.path.append('..')
sys.path.append('../..')


from pyspark import SparkContext
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    when,
    countDistinct,
    mean,
    desc,
    sum,
    lit,
    col,
    to_date,
)
from pyspark.sql.types import LongType

from lib_cf.cf_io import (
    load_config,
    calculate_filepaths,
)
from lib_cf.cf_utils import (
    member_in_range,
    join_category_details,
)
from lib.utils import trips_only

In [0]:
config_path = dbutils.widgets.get("config_path")
test = False if not dbutils.widgets.get("test") else True
rmse = False if not dbutils.widgets.get("rmse") else True
future = False if not dbutils.widgets.get("future") else True

In [0]:
# (1) --- Handle Arguments --- #
CNF, CFG_PATH = load_config(config_path, test, future, rmse)
PARAMS = dict(list(CNF["shared"].items()) + list(CNF["etl"].items()))
PATHS = CNF["paths"]
PARAMS, PATHS = calculate_filepaths(PARAMS, PATHS)

In [0]:
# (2) --- Read Data --- #
print(" 1/13 Reading Data...")
if PARAMS["test"]: # Test logic and paths will remain as is.
    memberDF = spark.read.csv(PATHS["CUBE"], header=True)
    transactionDF = spark.read.csv(PATHS["TRANS_INT"], header=True)
    itemDF = spark.read.csv(PATHS["ITEM_INT"], header=True)
else:
    memberDF = spark.table(fs_customer_cube_full)
    transactionDF = spark.table(silver_transaction_fiscal_detail)
    itemDF = spark.table(silver_ad_hoc_master_item_with_brand)

In [0]:
# (4) --- Prep Member Data --- #
print("2/13 Prepping Members...")
members = member_in_range(memberDF, PARAMS["start"], PARAMS["end"])
members = members.withColumn("MBRSHP_SID", members.MBRSHP_SID.cast(LongType()))
if PARAMS["sample"] < 1:
    members = members.sample(False, PARAMS["sample"], 42)
n_memb = members.count()

In [0]:
# (5) --- Prep Transaction Details --- #

print("3/13 Prepping Transactions...")
details = transactionDF[transactionDF.PURCH_DT >= PARAMS["start"]]
details = details[details.PURCH_DT <= PARAMS["end"]]
details = trips_only(details)
details = details.withColumn("MBRSHP_SID", details.MBRSHP_SID.cast(LongType()))
if PARAMS["sample"] < 1:
    details = details.sample(False, PARAMS["sample"], 42)
n_trans = details.count()

In [0]:
# (6) --- Prep Items --- #
print("4/13 Prepping Items...")
items = itemDF
if PARAMS["category"] == "BRAND":
    items = items.where(col("BRAND_DESC") != "UNBRANDED")
n_items = items.count()

print("proceeding to tranform data for ", n_memb, " members")
print("and ", n_items, " items")
print("with ", n_trans, " transactions")

In [0]:
# (8) --- Join and Aggregate --- #

print("5/13 Generating Aggregate Data...")
member_trans = members.join(details, "MBRSHP_SID", "inner")
joined = join_category_details(items, member_trans, PARAMS["category"])
aggregated = joined.groupBy("MBRSHP_SID", "CATEGORY_NAME", "CATEGORY_ID").agg(
    countDistinct(joined.PURCH_HDR_ID).alias("TRIPS"),
    sum(joined.NORMAL_PRC_AMT).alias("SALES_AMT"),
    sum(joined.QTY_IN_UNITS).alias("SALES_UNITS"),
)

aggregated = aggregated.withColumn(
    "MBRSHP_SID", aggregated.MBRSHP_SID.cast(LongType())
)
aggregated = aggregated.withColumn(
    "CATEGORY_ID", aggregated.CATEGORY_ID.try_cast(LongType())
)
aggregated = aggregated[
    aggregated.CATEGORY_NAME != "UNKNOWN"
]  # remove aggregate 'unknown' category
# remove aggregate 'unbranded' category
aggregated = aggregated[aggregated.CATEGORY_NAME != "UNBRANDED"]
aggregated = aggregated.fillna(0)

In [0]:
# (9) ---- Generate Trips by Category --- #

print("6/13 Generating Metrics by Category...")
bycat = aggregated.groupBy("CATEGORY_ID").agg(
    sum("TRIPS").alias("TRIPS"),
    sum("SALES_AMT").alias("SALES_AMT"),
    sum("SALES_UNITS").alias("SALES_UNITS"),
)
bycat = bycat.fillna(0)

print("7/13 Writing Metrics By Category")
save_cf_tables(bycat, cf_data_by_cat, PARAMS)

In [0]:
# (10) --- Generate Member-Category-Matrix --- #

print("8/13 Generating Member-Category-Matrix...")
# for now, always take top categories by sales dollars
top_cats = (
    bycat.orderBy(desc("SALES_AMT"))
    .limit(int(PARAMS["num_cats"]))
    .select("CATEGORY_ID")
    .collect()
)
top_cats = [str(i.CATEGORY_ID) for i in top_cats]
matrix = aggregated.select(
    ["MBRSHP_SID", "CATEGORY_ID", "TRIPS", "SALES_AMT", "SALES_UNITS"]
)
matrix = matrix[matrix.CATEGORY_ID.isin(top_cats)]
# remove 'UNBRANDED' categories, only appears in brand
matrix = matrix[matrix.CATEGORY_ID != "5265"]
matrix = matrix.fillna(0)
print("9/13 Writing Member-Category-Matrix...")
save_cf_tables(matrix, cf_matrix, PARAMS)

In [0]:
# (11) --- Generate Category Lookup --- #

print("10/13 Generating Category Name Lookup...")
catlookup = aggregated.select(["CATEGORY_NAME", "CATEGORY_ID"]).distinct()

print("11/13 Writing Category Name lookup...")
save_cf_tables(catlookup, cf_cat_lookup, PARAMS)

In [0]:
# (12) -- Generate full member-categories -- #

print("12/13 Generating Slate table...")
all_members = aggregated.select("MBRSHP_SID").distinct()
all_cats = matrix.select("CATEGORY_ID").distinct()
all_membercats = all_members.crossJoin(all_cats)

print("13/13 Writing Slate table...")
save_cf_tables(all_membercats, cf_slate, PARAMS)